# Global Sovereign Debt Crisis Monitor (2000-Present)

A yearly country-level panel covering sovereign debt, fiscal health, and crisis-risk signals
for ~180 countries from 2000 onwards. Updated automatically every week via GitHub Actions.

| Source | Coverage | Auth |
|--------|----------|------|
| IMF WEO (DataMapper API) | Debt/GDP, fiscal balance | None |
| World Bank Open Data | External debt, reserves, GDP, revenue, interest | None |
| FRED (St. Louis Fed) | US Treasury yields | Free key |
| Manual reference | S&P ratings, defaults, IMF programs | N/A |

## Setup

In [1]:
import os
import time
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

START_YEAR = 2000
END_YEAR   = datetime.now().year

IMF_SERIES = {
    'govt_debt_pct_gdp':      'GGXWDG_NGDP',  # gross govt debt % GDP
    'fiscal_balance_pct_gdp': 'GGXCNL_NGDP',  # net lending/borrowing % GDP
    '_gdp_usd_bn':            'NGDPD',          # nominal GDP USD bn
    '_population_mn':         'LP',             # population (millions)
    '_govt_revenue_imf':      'GGR_NGDP',       # govt revenue % GDP (IMF fallback)
    # REMOVED: interest_payments_pct_gdp (GGINT_NGDP) — returns 0 rows; column dropped
}

WB_SERIES = {
    'external_debt_usd_bn':    'DT.DOD.DECT.CD',     # external debt USD (→ billions)
    'foreign_reserves_usd_bn': 'FI.RES.TOTL.CD',     # reserves USD (→ billions)
    '_gdp_wb_usd':             'NY.GDP.MKTP.CD',      # GDP current USD
    '_population_wb':          'SP.POP.TOTL',          # population (persons)
    'govt_revenue_pct_gdp':    'GC.REV.XGRT.GD.ZS',  # revenue excl. grants % GDP
    # REMOVED: GC.XPN.INTP.GD.ZS — returns 0 rows; interest columns dropped entirely
}

# FRED columns (us_10yr_yield_pct, us_federal_debt_pct_gdp_fred) are also dropped —
# they had only 26 non-null rows out of 7 581 and require an API key not always available.
FRED_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    FRED_KEY = UserSecretsClient().get_secret('FRED_API_KEY') or ''
except Exception:
    FRED_KEY = os.environ.get('FRED_API_KEY', '')

print(f'Year range : {START_YEAR} - {END_YEAR}')
print(f'FRED key   : {"present" if FRED_KEY else "not set (FRED columns excluded from schema)"}')
print('All imports OK')


Year range : 2000 - 2026
FRED key   : present
All imports OK


## 1. IMF World Economic Outlook Data

Uses the IMF DataMapper API - no key required. Fetches gross debt (% GDP), fiscal balance, GDP, and population.
Revenue and interest payments come from World Bank in Section 2 (more reliable coverage).

- Endpoint: `https://www.imf.org/external/datamapper/api/v1/{SERIES_CODE}`
- Retries up to 3 times with backoff on network errors

In [2]:
def fetch_imf_series(series_code: str, col_name: str, retries: int = 3) -> pd.DataFrame:
    """Pull one IMF DataMapper series for all countries and years, with retries."""
    url  = f'https://www.imf.org/external/datamapper/api/v1/{series_code}'
    wait = 2
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=90)
            resp.raise_for_status()
            raw  = resp.json().get('values', {}).get(series_code, {})
            break
        except Exception as exc:
            print(f'    attempt {attempt}/{retries} failed for {series_code}: {exc}')
            if attempt == retries:
                return pd.DataFrame(columns=['iso3_code', 'year', col_name])
            time.sleep(wait)
            wait *= 2

    rows = []
    for iso3, year_vals in raw.items():
        if len(iso3) != 3 or not iso3.isalpha():
            continue
        for yr_str, val in year_vals.items():
            try:
                yr = int(yr_str)
            except ValueError:
                continue
            if START_YEAR <= yr <= END_YEAR and val is not None:
                rows.append({'iso3_code': iso3.upper(), 'year': yr, col_name: float(val)})

    if not rows:
        print(f'    WARNING: {series_code} returned no rows')
        return pd.DataFrame(columns=['iso3_code', 'year', col_name])
    return pd.DataFrame(rows)


def safe_merge(frames: dict) -> pd.DataFrame:
    """Outer-merge a dict of DataFrames on (iso3_code, year), skipping empty ones."""
    result = None
    for name, df in frames.items():
        if df.empty or 'iso3_code' not in df.columns or 'year' not in df.columns:
            print(f'    skipping empty frame: {name}')
            continue
        result = df if result is None else result.merge(df, on=['iso3_code','year'], how='outer')
    return result


print('Fetching IMF DataMapper series...')
imf_frames = {}
for col_name, code in IMF_SERIES.items():
    print(f'  {code:20s}  ->  {col_name}')
    imf_frames[col_name] = fetch_imf_series(code, col_name)
    time.sleep(0.5)

imf_panel = safe_merge(imf_frames)
if imf_panel is None or imf_panel.empty:
    raise RuntimeError('All IMF series came back empty - check network access on this platform.')

imf_panel = imf_panel.sort_values(['iso3_code','year']).reset_index(drop=True)
print(f'\nIMF panel: {len(imf_panel):,} rows | {imf_panel["iso3_code"].nunique()} countries')
imf_panel.head()

Fetching IMF DataMapper series...
  GGXWDG_NGDP           ->  govt_debt_pct_gdp
  GGXCNL_NGDP           ->  fiscal_balance_pct_gdp
  NGDPD                 ->  _gdp_usd_bn
  LP                    ->  _population_mn
  GGR_NGDP              ->  _govt_revenue_imf
    skipping empty frame: _govt_revenue_imf

IMF panel: 5,869 rows | 220 countries


,iso3_code,year,govt_debt_pct_gdp,fiscal_balance_pct_gdp,_gdp_usd_bn,_population_mn
0,ABW,2000,38.600,0.600,1.873,0.102
1,ABW,2001,44.300,-0.800,1.896,0.102
2,ABW,2002,47.100,-3.200,1.962,0.103
3,ABW,2003,40.800,1.900,2.044,0.105
4,ABW,2004,42.500,-8.500,2.255,0.106


## 2. World Bank Open Data

Provides external debt, reserves, GDP cross-check, population, and two fiscal series
(government revenue and interest payments) that fill in where IMF DataMapper is incomplete.

- Endpoint: `https://api.worldbank.org/v2/country/all/indicator/{CODE}?format=json`
- Automatically pages through all result pages

In [3]:
def fetch_wb_metadata() -> pd.DataFrame:
    """Get country name, ISO3, World Bank region, and income group.
    FIX: original code fetched only page 1 (missed ~70 countries); now paginates.
    """
    rows = []
    page = 1
    while True:
        url = (
            f'https://api.worldbank.org/v2/country'
            f'?format=json&per_page=500&page={page}'
        )
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        if not payload or len(payload) < 2:
            break
        for c in payload[1] or []:
            if c.get('region', {}).get('id') == 'NA':   # skip aggregate groups
                continue
            iso3 = c.get('id', '').upper()
            if not iso3:
                continue
            rows.append({
                'iso3_code':    iso3,
                'country':      c.get('name', ''),
                'region':       c.get('region', {}).get('value', ''),
                'income_group': c.get('incomeLevel', {}).get('value', ''),
            })
        if page >= payload[0].get('pages', 1):
            break
        page += 1
        time.sleep(0.3)
    return pd.DataFrame(rows)


def fetch_wb_indicator(indicator: str, col_name: str, retries: int = 3) -> pd.DataFrame:
    """Fetch one WB indicator for all countries and years, with pagination + retries."""
    rows = []
    page = 1
    while True:
        url = (
            f'https://api.worldbank.org/v2/country/all/indicator/{indicator}'
            f'?format=json&per_page=1000&date={START_YEAR}:{END_YEAR}&page={page}'
        )
        wait = 2
        payload = None
        for attempt in range(1, retries + 1):
            try:
                resp = requests.get(url, timeout=90)
                resp.raise_for_status()
                payload = resp.json()
                break
            except Exception as exc:
                print(f'    attempt {attempt}/{retries} WB {indicator} p{page}: {exc}')
                if attempt == retries:
                    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['iso3_code', 'year', col_name])
                time.sleep(wait); wait *= 2

        if not payload or len(payload) < 2 or not payload[1]:
            break
        for item in payload[1]:
            if item.get('value') is None:
                continue
            iso3 = (item.get('countryiso3code') or '').upper()
            if len(iso3) != 3:
                continue
            rows.append({'iso3_code': iso3, 'year': int(item['date']), col_name: item['value']})
        if page >= payload[0].get('pages', 1):
            break
        page += 1
        time.sleep(0.3)

    if not rows:
        return pd.DataFrame(columns=['iso3_code', 'year', col_name])
    return pd.DataFrame(rows)


print('Fetching World Bank metadata...')
wb_meta = fetch_wb_metadata()
print(f'  {len(wb_meta)} countries')

print('Fetching World Bank indicator series...')
wb_frames = {}
for col_name, indicator in WB_SERIES.items():
    print(f'  {indicator:35s}  ->  {col_name}')
    wb_frames[col_name] = fetch_wb_indicator(indicator, col_name)
    time.sleep(0.5)

wb_panel = safe_merge(wb_frames)
if wb_panel is None:
    raise RuntimeError('All World Bank series came back empty - check network access.')

# convert from raw USD to billions
for col in ['external_debt_usd_bn', 'foreign_reserves_usd_bn', '_gdp_wb_usd']:
    if col in wb_panel.columns:
        wb_panel[col] = wb_panel[col] / 1e9
if '_gdp_wb_usd' in wb_panel.columns:
    wb_panel = wb_panel.rename(columns={'_gdp_wb_usd': '_gdp_wb_usd_bn'})

wb_panel = wb_panel.sort_values(['iso3_code', 'year']).reset_index(drop=True)
print(f'\nWB panel: {len(wb_panel):,} rows | {wb_panel["iso3_code"].nunique()} countries')
wb_panel.head()


Fetching World Bank metadata...
  217 countries
Fetching World Bank indicator series...
  DT.DOD.DECT.CD                       ->  external_debt_usd_bn
    attempt 1/3 WB DT.DOD.DECT.CD p1: HTTPSConnectionPool(host='api.worldbank.org', port=443): Read timed out. (read timeout=90)
  FI.RES.TOTL.CD                       ->  foreign_reserves_usd_bn
  NY.GDP.MKTP.CD                       ->  _gdp_wb_usd
  SP.POP.TOTL                          ->  _population_wb
  GC.REV.XGRT.GD.ZS                    ->  govt_revenue_pct_gdp

WB panel: 6,525 rows | 261 countries


,iso3_code,year,external_debt_usd_bn,foreign_reserves_usd_bn,_gdp_wb_usd_bn,_population_wb,govt_revenue_pct_gdp
0,ABW,2000,NaN,0.235,1.873,90588,NaN
1,ABW,2001,NaN,0.321,1.896,91439,NaN
2,ABW,2002,NaN,0.374,1.962,92074,NaN
3,ABW,2003,NaN,0.337,2.044,93128,NaN
4,ABW,2004,NaN,0.339,2.255,95138,NaN


## 3. FRED Data (US-specific)

Adds US 10-year Treasury yield and federal debt-to-GDP from the St. Louis Fed.
Joined into the panel for the USA row only. Skipped gracefully if no key is set.

Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html

In [4]:
def fetch_fred_series(series_id: str, col_name: str, api_key: str, agg: str = 'avg') -> pd.DataFrame:
    """Pull one FRED series at annual frequency."""
    url = (
        f'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={api_key}&file_type=json'
        f'&observation_start={START_YEAR}-01-01&frequency=a&aggregation_method={agg}'
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    rows = []
    for obs in resp.json().get('observations', []):
        try:
            rows.append({'year': int(obs['date'][:4]), col_name: float(obs['value'])})
        except (ValueError, KeyError):
            pass
    df = pd.DataFrame(rows)
    if not df.empty:
        df['iso3_code'] = 'USA'
    return df


fred_panel = pd.DataFrame()

if FRED_KEY:
    print('FRED key found - fetching US series...')
    try:
        f1 = fetch_fred_series('DGS10',       'us_10yr_yield_pct',          FRED_KEY, 'avg')
        time.sleep(0.5)
        f2 = fetch_fred_series('GFDEGDQ188S', 'us_federal_debt_pct_gdp_fred', FRED_KEY, 'avg')
        fred_panel = f1.merge(f2, on=['iso3_code','year'], how='outer')
        print(f'  FRED: {len(fred_panel)} US rows')
    except Exception as exc:
        print(f'  FRED fetch failed: {exc} - skipping')
else:
    print('No FRED_API_KEY - skipping US FRED data')

FRED key found - fetching US series...
  FRED: 26 US rows


## 4. Static Reference Data

Three datasets with no public API - maintained by hand in the notebook:

- **S&P sovereign credit ratings** - current rating per country, mapped to numeric scale (AAA=21, D=0)
- **Sovereign default events** - known defaults from IMF reports and public records
- **IMF program flags** - years under an active IMF loan program (from IMF MONA database)

> Update these when new events happen - they do not refresh automatically.

In [5]:
RATING_MAP = {
    'AAA':21,'AA+':20,'AA':19,'AA-':18,
    'A+':17,'A':16,'A-':15,
    'BBB+':14,'BBB':13,'BBB-':12,
    'BB+':11,'BB':10,'BB-':9,
    'B+':8,'B':7,'B-':6,
    'CCC+':5,'CCC':4,'CCC-':3,
    'CC':2,'C':1,'D':0,'SD':0,'NR':np.nan,
}

CURRENT_SP_RATINGS = {
    'USA':'AA+','GBR':'AA', 'DEU':'AAA','FRA':'AA-','JPN':'A+',
    'CAN':'AAA','AUS':'AAA','CHE':'AAA','NLD':'AAA','SWE':'AAA',
    'NOR':'AAA','DNK':'AAA','FIN':'AA+','AUT':'AA+','BEL':'AA',
    'ITA':'BBB','ESP':'A',  'PRT':'BBB+','GRC':'BBB-','IRL':'AA',
    'CHN':'A+', 'KOR':'AA', 'SGP':'AAA','HKG':'AA+','TWN':'AA',
    'IND':'BBB-','IDN':'BBB','MYS':'A-','THA':'BBB+','PHL':'BBB+',
    'VNM':'BB+','BGD':'BB-','PAK':'CCC+','LKA':'SD','NPL':'NR',
    'BRA':'BB-','MEX':'BBB','COL':'BB+','PER':'BBB','CHL':'A-',
    'ARG':'CCC-','VEN':'SD','ECU':'B-','BOL':'B+','PRY':'BB',
    'URY':'BBB','PAN':'BBB','CRI':'B+','DOM':'BB','JAM':'B+',
    'ZAF':'BB-','EGY':'B','NGA':'B-','KEN':'B','GHA':'SD',
    'ETH':'SD', 'AGO':'B-','CIV':'BB-','CMR':'B','TZA':'NR',
    'UGA':'B+','SEN':'B+','MAR':'BB+','TUN':'CCC','DZA':'NR',
    'MOZ':'CCC+','ZMB':'SD','RWA':'B+','MLI':'NR','SDN':'NR',
    'RUS':'SD','TUR':'B+','POL':'A-','CZE':'AA-','HUN':'BBB',
    'ROU':'BBB-','BGR':'BBB','HRV':'BBB+','SVN':'AA-','SVK':'A+',
    'UKR':'SD','KAZ':'BBB','AZE':'BB+','UZB':'BB-','GEO':'BB',
    'SAU':'A','ARE':'AA','QAT':'AA','KWT':'AA','OMN':'BB+',
    'BHR':'B+','JOR':'B+','ISR':'AA-','LBN':'SD','IRQ':'B-',
    'NZL':'AA','ISL':'A','CYP':'BBB','MLT':'A-','EST':'AA-',
    'LVA':'A+','LTU':'A','LUX':'AAA',
}

years_range = list(range(START_YEAR, END_YEAR + 1))
rating_rows = [
    {'iso3_code': iso3, 'year': yr,
     'sp_credit_rating': rtg,
     'sp_rating_numeric': RATING_MAP.get(rtg, np.nan)}
    for iso3, rtg in CURRENT_SP_RATINGS.items()
    for yr in years_range
]
ratings_df = pd.DataFrame(rating_rows)
print(f'Ratings: {ratings_df["iso3_code"].nunique()} countries')

Ratings: 103 countries


In [6]:
DEFAULT_EVENTS = [
    ('ARG',2001),('ARG',2014),('ARG',2020),
    ('RUS',2022),('GRC',2012),
    ('ECU',2008),('ECU',2020),('JAM',2010),
    ('BLZ',2006),('BLZ',2012),('BLZ',2017),('BLZ',2021),
    ('ZMB',2020),('SRB',2004),('MDG',2002),('CMR',2004),
    ('ETH',2023),('GHA',2022),('LKA',2022),
    ('VEN',2017),('VEN',2018),('URY',2003),('UKR',2015),
    ('CIV',2000),('CIV',2011),('LBN',2020),('MOZ',2017),
    ('SUR',2020),('PAK',2023),
]
defaults_df = pd.DataFrame(DEFAULT_EVENTS, columns=['iso3_code','year'])
defaults_df['sovereign_default_event'] = 1

IMF_PROGRAMS = [
    ('ARG',2000,2002),('ARG',2018,2020),('ARG',2022,2025),
    ('GRC',2010,2012),('GRC',2012,2018),
    ('PRT',2011,2014),('IRL',2010,2013),
    ('UKR',2014,2016),('UKR',2018,2019),('UKR',2022,2025),
    ('PAK',2019,2024),
    ('EGY',2016,2019),('EGY',2022,2025),
    ('LKA',2023,2026),('GHA',2023,2026),('ETH',2024,2027),
    ('ZMB',2022,2025),
    ('TUN',2013,2015),('TUN',2023,2024),
    ('JAM',2010,2012),('JAM',2013,2016),('JAM',2019,2021),
    ('KEN',2021,2023),('SEN',2021,2023),('CIV',2023,2025),
    ('ECU',2019,2022),('CRI',2021,2023),
    ('BOL',2003,2004),('IDN',2000,2003),('RUS',2000,2002),
    ('TUR',2002,2008),('BRA',2003,2005),
    ('ROU',2009,2011),('ROU',2011,2013),('HUN',2008,2010),
]
program_rows = [
    {'iso3_code': iso3, 'year': yr, 'imf_program_active': 1}
    for iso3, s, e in IMF_PROGRAMS
    for yr in range(max(s, START_YEAR), min(e, END_YEAR) + 1)
]
programs_df = pd.DataFrame(program_rows).drop_duplicates()
print(f'Default events : {len(defaults_df)} records')
print(f'IMF programs   : {len(programs_df)} country-year records')

Default events : 29 records
IMF programs   : 123 country-year records


## 5. Build the Main Panel

Merge all sources on `(iso3_code, year)`, attach country metadata,
then compute the derived columns (debt per capita, ratios, risk score).

In [7]:
panel = imf_panel.copy()
panel = panel.merge(wb_panel,    on=['iso3_code', 'year'], how='outer')
panel = panel.merge(wb_meta,     on='iso3_code',           how='left')
panel = panel.merge(ratings_df,  on=['iso3_code', 'year'], how='left')
panel = panel.merge(defaults_df, on=['iso3_code', 'year'], how='left')
panel['sovereign_default_event'] = panel['sovereign_default_event'].fillna(0).astype(int)
panel = panel.merge(programs_df, on=['iso3_code', 'year'], how='left')
panel['imf_program_active'] = panel['imf_program_active'].fillna(0).astype(int)

# FRED merge 
panel = panel[
    panel['iso3_code'].str.match(r'^[A-Za-z]{3}$', na=False) &
    panel['year'].between(START_YEAR, END_YEAR)
].copy().sort_values(['iso3_code', 'year']).reset_index(drop=True)

if panel.empty:
    raise RuntimeError(
        'Panel is empty after merging — both IMF and World Bank fetches likely failed. '
        'Check the network access log above.'
    )

print(f'Panel rows   : {len(panel):,}')
print(f'Countries    : {panel["iso3_code"].nunique()}')
panel.head(3)


Panel rows   : 7,581
Countries    : 286


,iso3_code,year,govt_debt_pct_gdp,fiscal_balance_pct_gdp,_gdp_usd_bn,_population_mn,external_debt_usd_bn,foreign_reserves_usd_bn,_gdp_wb_usd_bn,_population_wb,govt_revenue_pct_gdp,country,region,income_group,sp_credit_rating,sp_rating_numeric,sovereign_default_event,imf_program_active
0,ABW,2000,38.600,0.600,1.873,0.102,NaN,0.235,1.873,90588.000,NaN,Aruba,Latin America & Caribbean,High income,NaN,NaN,0,0
1,ABW,2001,44.300,-0.800,1.896,0.102,NaN,0.321,1.896,91439.000,NaN,Aruba,Latin America & Caribbean,High income,NaN,NaN,0,0
2,ABW,2002,47.100,-3.200,1.962,0.103,NaN,0.374,1.962,92074.000,NaN,Aruba,Latin America & Caribbean,High income,NaN,NaN,0,0


In [8]:
# --- fallback fetches if World Bank sparse indicators came back empty ---
# WB GC.XPN.INTP.GD.ZS and GC.REV.XGRT.GD.ZS have limited historical coverage.
# If either is all-NaN in the panel, try the equivalent IMF DataMapper series.

if 'interest_payments_pct_gdp' not in panel.columns or panel.get('interest_payments_pct_gdp', pd.Series(dtype=float)).isna().all():
    print('  interest_payments empty - trying IMF GGXI_NGDP...')
    _fb = fetch_imf_series('GGXI_NGDP', 'interest_payments_pct_gdp')
    if not _fb.empty and 'interest_payments_pct_gdp' in _fb.columns:
        panel = panel.drop(columns=['interest_payments_pct_gdp'], errors='ignore')
        panel = panel.merge(_fb, on=['iso3_code', 'year'], how='left')
        print(f'    filled: {panel["interest_payments_pct_gdp"].notna().sum()} rows')

if 'govt_revenue_pct_gdp' not in panel.columns or panel.get('govt_revenue_pct_gdp', pd.Series(dtype=float)).isna().all():
    print('  govt_revenue empty - trying IMF GGR_NGDP...')
    _fb = fetch_imf_series('GGR_NGDP', 'govt_revenue_pct_gdp')
    if not _fb.empty and 'govt_revenue_pct_gdp' in _fb.columns:
        panel = panel.drop(columns=['govt_revenue_pct_gdp'], errors='ignore')
        panel = panel.merge(_fb, on=['iso3_code', 'year'], how='left')
        print(f'    filled: {panel["govt_revenue_pct_gdp"].notna().sum()} rows')

# Guarantee every expected source column exists before we touch it
EXPECTED = [
    'govt_debt_pct_gdp', 'fiscal_balance_pct_gdp',
    'govt_revenue_pct_gdp', 'interest_payments_pct_gdp',
    '_gdp_usd_bn', '_population_mn',
    'external_debt_usd_bn', 'foreign_reserves_usd_bn',
    '_gdp_wb_usd_bn', '_population_wb',
    '_govt_revenue_imf',
]
for col in EXPECTED:
    if col not in panel.columns:
        panel[col] = np.nan

# govt_revenue_pct_gdp: prefer WB, fill gaps with IMF GGR_NGDP
panel['govt_revenue_pct_gdp'] = panel['govt_revenue_pct_gdp'].combine_first(
    panel['_govt_revenue_imf']
)

# Best available GDP (prefer IMF, fall back to World Bank)
panel['_gdp_bn'] = panel['_gdp_usd_bn'].combine_first(panel['_gdp_wb_usd_bn'])

# Population as actual persons (IMF = millions, WB = persons)
panel['_pop'] = panel['_population_mn'].mul(1e6).combine_first(panel['_population_wb'])

# govt_debt_usd_bn: derived from debt% × GDP
panel['govt_debt_usd_bn'] = np.where(
    panel['govt_debt_pct_gdp'].notna() & panel['_gdp_bn'].notna(),
    panel['govt_debt_pct_gdp'] / 100 * panel['_gdp_bn'],
    np.nan,
)

# Debt per capita (USD)
panel['debt_per_capita_usd'] = np.where(
    panel['govt_debt_usd_bn'].notna() & panel['_pop'].notna() & (panel['_pop'] > 0),
    panel['govt_debt_usd_bn'] * 1e9 / panel['_pop'],
    np.nan,
)

# External debt % GDP
panel['external_debt_pct_gdp'] = np.where(
    panel['external_debt_usd_bn'].notna() & panel['_gdp_bn'].notna() & (panel['_gdp_bn'] > 0),
    panel['external_debt_usd_bn'] / panel['_gdp_bn'] * 100,
    np.nan,
)

# Reserves / external debt ratio
panel['reserves_to_external_debt'] = np.where(
    panel['foreign_reserves_usd_bn'].notna() & panel['external_debt_usd_bn'].notna()
    & (panel['external_debt_usd_bn'] > 0),
    panel['foreign_reserves_usd_bn'] / panel['external_debt_usd_bn'],
    np.nan,
)

# Composite risk score 0-100

debt_score = (panel['govt_debt_pct_gdp'].clip(0, 200) / 200 * 100).fillna(50)
res_raw    = panel['reserves_to_external_debt'].clip(0, 2).fillna(0.5)
res_score  = (1 - res_raw / 2) * 100
panel['debt_crisis_risk_score'] = (0.60 * debt_score + 0.40 * res_score).round(2)

# Drop internal helper columns
panel = panel.drop(columns=[c for c in panel.columns if c.startswith('_')])

# Final column order — only columns with real data
id_cols     = ['year', 'country', 'iso3_code', 'region', 'income_group']
debt_cols   = ['govt_debt_pct_gdp', 'govt_debt_usd_bn', 'external_debt_usd_bn',
               'external_debt_pct_gdp', 'debt_per_capita_usd']
fiscal_cols = ['fiscal_balance_pct_gdp', 'govt_revenue_pct_gdp',
               'foreign_reserves_usd_bn', 'reserves_to_external_debt']
risk_cols   = ['sp_credit_rating', 'sp_rating_numeric', 'imf_program_active',
               'sovereign_default_event', 'debt_crisis_risk_score']

all_cols = id_cols + debt_cols + fiscal_cols + risk_cols
panel    = panel[[c for c in all_cols if c in panel.columns]]

print(f'Final panel: {len(panel):,} rows | {panel.shape[1]} columns')
panel.describe()


Final panel: 7,581 rows | 19 columns


,year,govt_debt_pct_gdp,govt_debt_usd_bn,external_debt_usd_bn,external_debt_pct_gdp,debt_per_capita_usd,fiscal_balance_pct_gdp,govt_revenue_pct_gdp,foreign_reserves_usd_bn,reserves_to_external_debt,sp_rating_numeric,imf_program_active,sovereign_default_event,debt_crisis_risk_score
count,7581.000,5713.000,5712.000,3182.000,3167.000,5712.000,5778.000,3627.000,4343.000,2613.000,2643.000,7581.000,7581.000,7581.000
mean,2012.767,55.251,1044.765,162.989,50.232,8118.904,-2.101,25.815,60.007,0.725,12.555,0.016,0.004,49.690
std,7.670,42.223,4736.825,741.168,43.325,14321.488,6.422,14.914,245.033,2.213,6.138,0.125,0.062,13.426
min,2000.000,0.000,0.000,0.070,1.093,0.000,-55.700,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,2006.000,30.000,2.655,1.974,24.393,651.808,-4.500,17.453,0.716,0.198,8.000,0.000,0.000,41.100
50%,2013.000,46.800,14.895,7.637,39.321,2120.833,-2.400,24.575,4.035,0.340,13.000,0.000,0.000,49.600
75%,2019.000,69.800,156.842,35.264,64.125,8327.969,-0.200,33.011,28.512,0.582,18.000,0.000,0.000,60.000
max,2026.000,600.100,68433.838,8935.869,509.509,185231.869,125.100,344.999,3900.039,38.391,21.000,1.000,1.000,100.000


## 6. ML Feature Engineering

Builds a model-ready feature matrix from the panel with:
- **Lags**: 1-year and 2-year lag of each numeric column
- **Rolling windows**: 3-year and 5-year rolling mean, 3-year rolling std
- **Year-over-year**: absolute change and % change (capped at ±200%)
- **Target**: `target_risk_rise_next_yr` = 1 if `debt_crisis_risk_score` rises >5 pts next year

In [9]:
ALL_FEAT = [
    'govt_debt_pct_gdp', 'fiscal_balance_pct_gdp',
    'govt_revenue_pct_gdp',
    'external_debt_pct_gdp', 'debt_per_capita_usd',
    'foreign_reserves_usd_bn', 'reserves_to_external_debt',
    'debt_crisis_risk_score', 'sp_rating_numeric',
    # REMOVED: primary_balance_pct_gdp, interest_payments_pct_gdp,
    #          interest_pct_revenue — all had 0 non-null rows
]
FEAT_COLS = [c for c in ALL_FEAT if c in panel.columns]
print(f'Building features for {len(FEAT_COLS)} columns')

ml = panel.sort_values(['iso3_code', 'year']).copy()

for col in FEAT_COLS:
    g = ml.groupby('iso3_code')[col]

    ml[f'{col}_lag1']       = g.shift(1)
    ml[f'{col}_lag2']       = g.shift(2)
    ml[f'{col}_yoy_change'] = g.diff(1)

    # transform('pct_change') avoids FutureWarning from GroupBy.pct_change(n)
    ml[f'{col}_yoy_pct'] = g.transform('pct_change').clip(-2, 2) * 100

    # Compute rolling on pre-shifted lag1 to avoid FutureWarning in pandas 2.x
    lag1 = ml[f'{col}_lag1']
    ml[f'{col}_roll3_mean'] = lag1.groupby(ml['iso3_code']).transform(lambda x: x.rolling(3).mean())
    ml[f'{col}_roll5_mean'] = lag1.groupby(ml['iso3_code']).transform(lambda x: x.rolling(5).mean())
    ml[f'{col}_roll3_std']  = lag1.groupby(ml['iso3_code']).transform(lambda x: x.rolling(3).std())

if 'debt_crisis_risk_score' in ml.columns:
    nxt = ml.groupby('iso3_code')['debt_crisis_risk_score'].shift(-1)
    ml['target_risk_rise_next_yr'] = ((nxt - ml['debt_crisis_risk_score']) > 5).astype('Int8')
else:
    ml['target_risk_rise_next_yr'] = pd.NA

ml = ml.sort_values(['iso3_code', 'year']).reset_index(drop=True)
print(f'ML matrix: {len(ml):,} rows | {ml.shape[1]} columns')
print(f'New feature cols: {ml.shape[1] - panel.shape[1]}')
if ml['target_risk_rise_next_yr'].notna().any():
    print('\nTarget distribution:')
    print(ml['target_risk_rise_next_yr'].value_counts())


Building features for 9 columns


/tmp/ipykernel_57/2877228846.py:23: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ml[f'{col}_yoy_pct'] = g.transform('pct_change').clip(-2, 2) * 100
/tmp/ipykernel_57/2877228846.py:23: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ml[f'{col}_yoy_pct'] = g.transform('pct_change').clip(-2, 2) * 100
/tmp/ipykernel_57/2877228846.py:23: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values

ML matrix: 7,581 rows | 83 columns
New feature cols: 64

Target distribution:
target_risk_rise_next_yr
0    7337
1     244
Name: count, dtype: Int64


/tmp/ipykernel_57/2877228846.py:23: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ml[f'{col}_yoy_pct'] = g.transform('pct_change').clip(-2, 2) * 100


## 7. Save Outputs

Writes both CSVs to the working directory and prints a row/coverage summary
so the GitHub Actions log shows exactly what was produced.

In [10]:
panel.to_csv('sovereign_debt_panel.csv', index=False)
ml.to_csv('ml_features.csv', index=False)

valid_years = panel['year'].dropna()
yr_min = int(valid_years.min()) if not valid_years.empty else START_YEAR
yr_max = int(valid_years.max()) if not valid_years.empty else END_YEAR

def col_coverage(df, col):
    """Return non-null row count for a column, or 'missing' if the column isn't there."""
    return f'{df[col].notna().sum()} rows' if col in df.columns else 'column missing'

print('--- sovereign_debt_panel.csv ---')
print(f'  rows      : {len(panel):,}')
print(f'  columns   : {panel.shape[1]}')
print(f'  countries : {panel["iso3_code"].nunique()}')
print(f'  years     : {yr_min} - {yr_max}')
print(f'  debt/GDP  : {col_coverage(panel, "govt_debt_pct_gdp")}')
print(f'  revenue   : {col_coverage(panel, "govt_revenue_pct_gdp")}')
print(f'  interest  : {col_coverage(panel, "interest_payments_pct_gdp")}')
print(f'  reserves  : {col_coverage(panel, "foreign_reserves_usd_bn")}')
print()
print('--- ml_features.csv ---')
print(f'  rows    : {len(ml):,}')
print(f'  columns : {ml.shape[1]}')
print(f'  target (not null): {ml["target_risk_rise_next_yr"].notna().sum()}')

--- sovereign_debt_panel.csv ---
  rows      : 7,581
  columns   : 19
  countries : 286
  years     : 2000 - 2026
  debt/GDP coverage  : 5713 rows
  revenue coverage   : 3627 rows
  reserves coverage  : 4343 rows

--- ml_features.csv ---
  rows    : 7,581
  columns : 83
  target (not null): 7581
